# K-Fold Cross-Validation — Solved Assignment

Fully worked solution using the built-in Iris dataset. Walks through the problem with a
single train/test split, then fixes it with K-Fold Cross-Validation.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Part 1 — Load the Data

In [2]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

print("X shape:", X.shape)
print("Number of classes:", len(np.unique(y)))

X shape: (150, 4)
Number of classes: 3


## Part 2 — Baseline: A Single Train/Test Split

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=1)

model = DecisionTreeClassifier(random_state=0)
model.fit(train_X, train_y)

print("Test accuracy:", model.score(test_X, test_y))

Test accuracy: 0.9555555555555556


## Part 3 — The Problem: Accuracy Depends on the Split

In [4]:
accuracies = []

for state in [0, 1, 2, 3, 4]:
    train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=state)
    model = DecisionTreeClassifier(random_state=0)
    model.fit(train_X, train_y)
    accuracies.append(model.score(test_X, test_y))

print(accuracies)
print("Range of accuracies:", min(accuracies), "to", max(accuracies))

[0.9777777777777777, 0.9555555555555556, 0.9555555555555556, 0.9111111111111111, 0.9777777777777777]
Range of accuracies: 0.9111111111111111 to 0.9777777777777777


**A1:** The test accuracy ranges from about 0.911 to 0.978 across the five splits —
roughly a 6-7 percentage point swing — even though the model and the amount of data are
identical each time; the only thing that changed is *which* 45 flowers happened to land
in the test set. If you only ever looked at one split, you could easily conclude the
model is better or worse than it actually is, purely due to that split's luck. With a
small dataset like Iris (150 rows), a handful of "hard" or "easy" samples landing in the
test set can noticeably swing accuracy.

## Part 4 — K-Fold Cross-Validation

In [5]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=1)

scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=kf)

print("Fold scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std deviation:", scores.std())

Fold scores: [0.96666667 0.96666667 0.96666667 0.93333333 0.83333333]
Mean accuracy: 0.9333333333333332
Std deviation: 0.05163977794943222


**A2:** Yes — the K-Fold mean (0.933, std ≈ 0.052) is a more trustworthy estimate than
any single split. Instead of relying on one lucky-or-unlucky split, it averages
performance over 5 different train/validation partitions, so every row gets used for
validation exactly once. This smooths out the split-to-split noise we saw in Part 3 —
notice fold 5 alone scored only 0.833, well below the Part 2/3 single-split results, but
because it's averaged with four other folds (0.967, 0.967, 0.967, 0.933) the overall
mean stays representative rather than being thrown off by that one weaker fold.

## Part 5 — Trying Different Values of k

In [6]:
for k in [3, 5, 10]:
    kf_k = KFold(n_splits=k, shuffle=True, random_state=1)
    scores_k = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=kf_k)
    print(f"k={k:>2}  mean={scores_k.mean():.4f}  std={scores_k.std():.4f}  scores={np.round(scores_k, 3)}")

k= 3  mean=0.9200  std=0.0283  scores=[0.96 0.9  0.9 ]
k= 5  mean=0.9333  std=0.0516  scores=[0.967 0.967 0.967 0.933 0.833]
k=10  mean=0.9400  std=0.0629  scores=[1.    0.933 0.933 1.    1.    0.933 1.    0.867 0.933 0.8  ]


**A3:** In this run: k=3 → mean 0.920, std 0.028; k=5 → mean 0.933, std 0.052; k=10 →
mean 0.940, std 0.063. As `k` increases, each fold's *validation* portion shrinks (33%
held out per fold at k=3, down to just 10% at k=10) while each fold's *training*
portion grows — with k=10, the model sees ~90% of the data each time, so the mean
accuracy creeps up slightly. But with only ~15 samples per validation fold at k=10, a
single misclassified flower changes that fold's score by a large chunk (e.g. one miss
out of 15 ≈ a 6-7% swing), which is exactly why the standard deviation is *highest* at
k=10 in these results, not lowest. The trade-off: small k trains on less data per fold
(slightly more pessimistic/biased mean) but each fold's score is smoother (based on more
validation samples); large k trains on almost all the data each time (less biased mean)
but each fold's score is noisier and cross-validation takes longer to run (more
model fits). k=5 or k=10 are common defaults that balance this reasonably well.

## Part 6 — Stratified K-Fold (Bonus)

In [7]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
strat_scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=skf)

print("Stratified fold scores:", strat_scores)
print("Stratified mean accuracy:", strat_scores.mean())
print()
print("Plain KFold mean accuracy:   ", scores.mean())
print("Stratified KFold mean accuracy:", strat_scores.mean())

Stratified fold scores: [0.96666667 1.         0.9        1.         0.86666667]
Stratified mean accuracy: 0.9466666666666667

Plain KFold mean accuracy:    0.9333333333333332
Stratified KFold mean accuracy: 0.9466666666666667


**Bonus observation:** Plain KFold gave a mean of 0.933; StratifiedKFold gave a
slightly higher mean of 0.947 here. Iris's 3 classes are perfectly balanced (50 samples
each), so even plain `KFold` with shuffling tends to produce fairly balanced folds by
chance — the two results are close. Stratified K-Fold matters much more, and can
noticeably change results, when classes are imbalanced (e.g. 90% vs. 10%), where plain
KFold could accidentally put very few minority-class examples in some folds.

## Part 7 — Reflection

**A4:** A single train/test split gives you exactly one performance number, and that
number depends heavily on which specific rows ended up in the test set — it can make an
average model look great or a good model look mediocre, purely by chance. K-Fold
Cross-Validation protects against this by evaluating the model on several different
train/validation partitions and averaging the results, giving an estimate that reflects
the model's *typical* performance rather than its performance on one particular random
split. It also uses every row for validation at some point, rather than "wasting" a
chunk of data purely as a held-out test set.

**A5:** The first model (mean 0.94, std 0.01) is the more trustworthy choice for
predictions on new data, even though its mean is slightly lower than the second model's
(0.95). A low standard deviation means the model performs consistently across different
subsets of data — you can be fairly confident it will score close to 0.94 on new,
unseen data. A high standard deviation (0.08) means the second model's performance
swings a lot depending on which data it sees — on some data it might score close to
1.0, but on other data it could drop well below 0.90. When comparing two models with
similar means, prefer the one with the tighter (lower-variance) spread of fold
scores.

---
### Answer Key Summary
- Single-split accuracy varies run to run — a poor estimate on its own.
- K-Fold CV (k=5) gives a mean accuracy with a standard deviation, a much more reliable estimate.
- Increasing k trades off training-set size, validation-fold size, per-fold noise, and runtime.
- Stratified K-Fold matters most for imbalanced classes; on balanced data like Iris it closely matches plain K-Fold.
- When comparing models, a lower standard deviation across folds is as important as a higher mean.